Clustering is good (modularity via training), but what if we just force it right from the start and train disjoint components (experts?) from the get go?

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import tqdm
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST, CIFAR10
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import mutual_info_score
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from IPython.display import clear_output
from collections import defaultdict
from itertools import islice
import random
import time
from pathlib import Path
import math

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

def randomseed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(3072, 64, bias=False)
        self.fc2 = nn.Linear(64, 64, bias=False)
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = x.view(-1, 3072)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
def accuracy(model, data):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data:
            outputs = model(images.to(device))
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels.to(device)).sum().item()
    return correct / total

def classwise_accuracy(model, data):
    model.eval()
    correct = defaultdict(int)
    total = defaultdict(int)
    with torch.no_grad():
        for images, labels in data:
            outputs = model(images.to(device))
            _, predicted = torch.max(outputs.data, 1)
            for i in range(len(labels)):
                label = labels[i].item()
                total[label] += 1
                correct[label] += int(predicted[i] == label)
    return [round(correct[i] / total[i], 3) if total[i] > 0 else 0 for i in range(10)]

train_dataset = CIFAR10(root='.', train=True, download=True, transform=torchvision.transforms.ToTensor())
test_dataset = CIFAR10(root='.', train=False, download=True, transform=torchvision.transforms.ToTensor())

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

def cluster_goodness_fast(model, cluster_U_indices, cluster_V_indices, num_clusters):
    A = model.fc2.weight ** 2
    mask = torch.zeros_like(A, dtype=torch.bool)
    
    for cluster_idx in range(num_clusters):
        u_indices = torch.tensor(cluster_U_indices[cluster_idx], dtype=torch.long)
        v_indices = torch.tensor(cluster_V_indices[cluster_idx], dtype=torch.long)
        mask[u_indices.unsqueeze(1), v_indices] = True
    
    intra_cluster_out_sum = torch.sum(A[mask])
    total_out_sum = torch.sum(A)
    
    return intra_cluster_out_sum / total_out_sum

Files already downloaded and verified
Files already downloaded and verified


In [2]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(16 * 16 * 16, 64, bias=False)
        self.fc2 = nn.Linear(64, 64, bias=False)
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 16 * 16 * 16)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [42]:
class CNNModified(nn.Module):
    def __init__(self):
        super(CNNModified, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(16 * 16 * 16, 64, bias=False)
        
        # Four separate fully connected layers for the output parts
        self.fc2_part1 = nn.Linear(16, 16, bias=False)
        self.fc2_part2 = nn.Linear(16, 16, bias=False)
        self.fc2_part3 = nn.Linear(16, 16, bias=False)
        self.fc2_part4 = nn.Linear(16, 16, bias=False)

        self.active_parts = [True, True, True, True]
        
        # Final fully connected layer
        self.fc3 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 16 * 16 * 16)
        
        # Pass through the first fully connected layer
        x = torch.relu(self.fc1(x))
        
        # Split into four parts
        x1 = x[:, :16]  # First 16 dimensions
        x2 = x[:, 16:32]  # Next 16 dimensions
        x3 = x[:, 32:48]  # Next 16 dimensions
        x4 = x[:, 48:]  # Last 16 dimensions
        
        # Pass through the four fc2 parts
        if self.active_parts[0]:
            x1 = self.fc2_part1(x1)
        else:
            x1 = torch.zeros_like(x1)
        if self.active_parts[1]:
            x2 = self.fc2_part2(x2)
        else:
            x2 = torch.zeros_like(x2)
        if self.active_parts[2]:
            x3 = self.fc2_part3(x3)
        else:
            x3 = torch.zeros_like(x3)
        if self.active_parts[3]:
            x4 = self.fc2_part4(x4)
        else:
            x4 = torch.zeros_like(x4)
        
        # Concatenate the outputs
        x = torch.cat((x1, x2, x3, x4), dim=1)
        
        # Pass through the final fully connected layer
        x = self.fc3(x)
        return x

In [37]:
model_unclustered = CNNModified()
model_unclustered.to(device)

# Train the model without clustering
optimizer = optim.Adam(model_unclustered.parameters(), lr=1e-3)
ce_losses_unclustered = []
epochs = 10

randomseed(42)

criterion = nn.CrossEntropyLoss()

In [38]:
for epoch in range(epochs):
    model_unclustered.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model_unclustered(data)
        loss = criterion(output, target)
        ce_losses_unclustered.append(loss.item())
        loss.backward()
        optimizer.step()
        if batch_idx % 900 == 0:
            acc = accuracy(model_unclustered, test_loader)
            print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}, Accuracy: {acc:.4f}')

Epoch 1/10, Batch 0/782, Loss: 2.3025, Accuracy: 0.1000
Epoch 2/10, Batch 0/782, Loss: 1.3881, Accuracy: 0.4633
Epoch 3/10, Batch 0/782, Loss: 1.5123, Accuracy: 0.5076
Epoch 4/10, Batch 0/782, Loss: 1.4325, Accuracy: 0.5454
Epoch 5/10, Batch 0/782, Loss: 1.2163, Accuracy: 0.5539
Epoch 6/10, Batch 0/782, Loss: 1.1225, Accuracy: 0.5846
Epoch 7/10, Batch 0/782, Loss: 0.9673, Accuracy: 0.5860
Epoch 8/10, Batch 0/782, Loss: 1.2460, Accuracy: 0.5964
Epoch 9/10, Batch 0/782, Loss: 1.3329, Accuracy: 0.6092
Epoch 10/10, Batch 0/782, Loss: 1.0963, Accuracy: 0.6055


In [39]:
classwise_accuracy(model_unclustered, test_loader)

[0.76, 0.725, 0.497, 0.452, 0.546, 0.554, 0.765, 0.587, 0.648, 0.69]

In [8]:
from copy import deepcopy

In [25]:
path: Path = Path("null_hypothesis/cifar10_cnn_modified.pth")
path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model_unclustered.state_dict(), path)

In [14]:
# load the model
model_unclustered = CNNModified()
model_unclustered.load_state_dict(torch.load(path))
model_unclustered.to(device)

CNNModified(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=64, bias=False)
  (fc2_part1): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part2): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part3): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part4): Linear(in_features=16, out_features=16, bias=False)
  (fc3): Linear(in_features=64, out_features=10, bias=False)
)

In [53]:
classwise_accuracy(model_unclustered, test_loader)

[0.76, 0.725, 0.497, 0.452, 0.546, 0.554, 0.765, 0.587, 0.648, 0.69]

In [56]:
def turn_off(model, part):
    # return the model with the part turned off
    model_copy = deepcopy(model)
    model_copy.active_parts[part] = False
    model_copy.eval()
    model_copy.to(device)
    return model_copy

def turn_on(model, part):
    # return the model with the part turned on
    model_copy = deepcopy(model)
    model_copy.active_parts[part] = True
    # everythign else is turned off
    for i in range(4):
        if i != part:
            model_copy.active_parts[i] = False
    model_copy.eval()
    model_copy.to(device)
    return model_copy

In [67]:
# plot the class-wise accuracies for each cluster turned off

num_clusters = 4

classwise_accuracies = []

for cluster_idx in tqdm.trange(num_clusters):
    model = CNNModified()
    model.load_state_dict(torch.load(path))
    model.to(device)
    model = turn_off(model, cluster_idx)
    classwise_accuracies.append(classwise_accuracy(model, test_loader))

100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


In [68]:
classwise_accuracies

[[0.415, 0.504, 0.475, 0.49, 0.542, 0.494, 0.656, 0.649, 0.747, 0.527],
 [0.732, 0.694, 0.634, 0.239, 0.148, 0.65, 0.505, 0.501, 0.295, 0.323],
 [0.697, 0.671, 0.57, 0.111, 0.429, 0.205, 0.828, 0.696, 0.598, 0.711],
 [0.58, 0.69, 0.101, 0.519, 0.627, 0.37, 0.771, 0.616, 0.628, 0.752]]

In [61]:
model = CNNModified()
model.load_state_dict(torch.load(path))
model.to(device)

CNNModified(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=64, bias=False)
  (fc2_part1): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part2): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part3): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part4): Linear(in_features=16, out_features=16, bias=False)
  (fc3): Linear(in_features=64, out_features=10, bias=False)
)

In [62]:
print(classwise_accuracy(model, test_loader))
# turn off first cluster
model = turn_on(model, 0)
print(classwise_accuracy(model, test_loader))
# turn off second cluster
model = turn_off(model, 1)
print(classwise_accuracy(model, test_loader))

[0.744, 0.72, 0.502, 0.403, 0.573, 0.511, 0.79, 0.698, 0.665, 0.672]
[0.393, 0.586, 0.336, 0.0, 0.076, 0.001, 0.779, 0.515, 0.003, 0.056]
[0.393, 0.586, 0.336, 0.0, 0.076, 0.001, 0.779, 0.515, 0.003, 0.056]


In [69]:
color_scale = pc.qualitative.G10  # You can use other scales like `pc.sequential.Plasma` or `pc.sequential.Viridis`
colors = color_scale[:num_clusters]  # Ensure we have enough colors for the number of clusters

# Create subplots: one row per cluster
fig = make_subplots(rows=num_clusters, cols=1, shared_xaxes=False, 
                    subplot_titles=[f'Cluster {i}' for i in range(num_clusters)])

for i in range(num_clusters):
    fig.add_trace(go.Bar(
        x=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
        y=classwise_accuracies[i],
        marker_color=colors[i],
        name=f'Cluster {i}',
    ), row=i+1, col=1)

# white background
fig.update_layout(plot_bgcolor='white')

# gridlines
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

fig.update_layout(height=150 * num_clusters, width=700, title_text='Class-wise accuracy with each cluster turned OFF.')

fig.show()

In [70]:
# plot the class-wise accuracies for each cluster turned off

num_clusters = 4

classwise_accuracies = []

for cluster_idx in tqdm.trange(num_clusters):
    model = CNNModified()
    model.load_state_dict(torch.load(path))
    model.to(device)
    model = turn_on(model, cluster_idx)
    classwise_accuracies.append(classwise_accuracy(model, test_loader))

100%|██████████| 4/4 [00:04<00:00,  1.09s/it]


In [22]:
model = CNNModified()
model.load_state_dict(torch.load(path))
model.to(device)

CNNModified(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=64, bias=False)
  (fc2_part1): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part2): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part3): Linear(in_features=16, out_features=16, bias=False)
  (fc2_part4): Linear(in_features=16, out_features=16, bias=False)
  (fc3): Linear(in_features=64, out_features=10, bias=False)
)

In [71]:
# Create subplots: one row per cluster

fig = make_subplots(rows=num_clusters, cols=1, shared_xaxes=False,
                    subplot_titles=[f'Cluster {i}' for i in range(num_clusters)])

for i in range(num_clusters):
    fig.add_trace(go.Bar(
        x=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
        y=classwise_accuracies[i],
        marker_color=colors[i],
        name=f'Cluster {i}',
    ), row=i+1, col=1)

# white background
fig.update_layout(plot_bgcolor='white')

# gridlines

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

fig.update_layout(height=150 * num_clusters, width=700, title_text='Class-wise accuracy with each cluster turned ON.')

fig.show()